<a href="https://colab.research.google.com/github/akashgardas/Deep-Learning/blob/feed-forward/models/feed_forward/Car_Price_Prediction_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Car Price Prediction

In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

# Data Preparation and Processing

## Load Dataset

In [3]:
path = '/content/drive/MyDrive/Tek works/Deep-Learning/Datasets/CarPrice_dataset.csv'
df = pd.read_csv(path)
df.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


## Inspect Data

In [ ]:
df.shape

(205, 26)

In [ ]:
df.columns

Index(['car_ID', 'symboling', 'CarName', 'fueltype', 'aspiration',
       'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'wheelbase',
       'carlength', 'carwidth', 'carheight', 'curbweight', 'enginetype',
       'cylindernumber', 'enginesize', 'fuelsystem', 'boreratio', 'stroke',
       'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg',
       'price'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   car_ID            205 non-null    int64  
 1   symboling         205 non-null    int64  
 2   CarName           205 non-null    object 
 3   fueltype          205 non-null    object 
 4   aspiration        205 non-null    object 
 5   doornumber        205 non-null    object 
 6   carbody           205 non-null    object 
 7   drivewheel        205 non-null    object 
 8   enginelocation    205 non-null    object 
 9   wheelbase         205 non-null    float64
 10  carlength         205 non-null    float64
 11  carwidth          205 non-null    float64
 12  carheight         205 non-null    float64
 13  curbweight        205 non-null    int64  
 14  enginetype        205 non-null    object 
 15  cylindernumber    205 non-null    object 
 16  enginesize        205 non-null    int64  
 1

In [ ]:
df.describe()

,car_ID,symboling,wheelbase,carlength,carwidth,carheight,curbweight,enginesize,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
count,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000
mean,103.000000,0.834146,98.756585,174.049268,65.907805,53.724878,2555.565854,126.907317,3.329756,3.255415,10.142537,104.117073,5125.121951,25.219512,30.751220,13276.710571
std,59.322565,1.245307,6.021776,12.337289,2.145204,2.443522,520.680204,41.642693,0.270844,0.313597,3.972040,39.544167,476.985643,6.542142,6.886443,7988.852332
min,1.000000,-2.000000,86.600000,141.100000,60.300000,47.800000,1488.000000,61.000000,2.540000,2.070000,7.000000,48.000000,4150.000000,13.000000,16.000000,5118.000000
25%,52.000000,0.000000,94.500000,166.300000,64.100000,52.000000,2145.000000,97.000000,3.150000,3.110000,8.600000,70.000000,4800.000000,19.000000,25.000000,7788.000000
50%,103.000000,1.000000,97.000000,173.200000,65.500000,54.100000,2414.000000,120.000000,3.310000,3.290000,9.000000,95.000000,5200.000000,24.000000,30.000000,10295.000000
75%,154.000000,2.000000,102.400000,183.100000,66.900000,55.500000,2935.000000,141.000000,3.580000,3.410000,9.400000,116.000000,5500.000000,30.000000,34.000000,16503.000000
max,205.000000,3.000000,120.900000,208.100000,72.300000,59.800000,4066.000000,326.000000,3.940000,4.170000,23.000000,288.000000,6600.000000,49.000000,54.000000,45400.000000


## Clean Data

In [ ]:
# Drop car_ID
df.drop('car_ID', axis=1, inplace=True)

In [ ]:
# Duplicates
df.duplicated().sum()

np.int64(0)

In [ ]:
# Missing values
df.isna().sum()

,0
symboling,0
CarName,0
fueltype,0
aspiration,0
doornumber,0
carbody,0
drivewheel,0
enginelocation,0
wheelbase,0
carlength,0


## Separate Dataset

In [5]:
X = df.drop('price', axis=1)
y = df['price']

## Preprocessing

In [8]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(categorical_cols)
print(numerical_cols)

['CarName', 'fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem']
['car_ID', 'symboling', 'wheelbase', 'carlength', 'carwidth', 'carheight', 'curbweight', 'enginesize', 'boreratio', 'stroke', 'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg']


In [16]:
def get_processed_df(preprocessor, X_original):
    """
    Converts the output of a fitted ColumnTransformer back into a Pandas DataFrame.
    """
    # 1. Get the new feature names from the fitted preprocessor
    feature_names = preprocessor.get_feature_names_out()

    # 2. Get the transformed array and convert to dense if it's sparse
    X_array = preprocessor.transform(X_original)
    if hasattr(X_array, 'toarray'): # Check if it's a sparse matrix
        X_array = X_array.toarray()

    # 3. Create the new DataFrame, keeping the original index
    return pd.DataFrame(X_array, columns=feature_names, index=X_original.index)

In [17]:
# Define Preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='drop'  # Explicitly drop columns not specified (or use 'passthrough')
)

# Apply Preprocessing
preprocessor.fit(X)
X_processed = get_processed_df(preprocessor, X)

In [11]:
X.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154,5000,19,26
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.40,10.0,102,5500,24,30
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.40,8.0,115,5500,18,22


In [18]:
X_processed.head()

,num__car_ID,num__symboling,num__wheelbase,num__carlength,num__carwidth,num__carheight,num__curbweight,num__enginesize,num__boreratio,num__stroke,num__compressionratio,num__horsepower,num__peakrpm,num__citympg,num__highwaympg,cat__CarName_Nissan versa,cat__CarName_alfa-romero Quadrifoglio,cat__CarName_alfa-romero giulia,cat__CarName_alfa-romero stelvio,cat__CarName_audi 100 ls,cat__CarName_audi 100ls,cat__CarName_audi 4000,cat__CarName_audi 5000,cat__CarName_audi 5000s (diesel),cat__CarName_audi fox,cat__CarName_bmw 320i,cat__CarName_bmw x1,cat__CarName_bmw x3,cat__CarName_bmw x4,cat__CarName_bmw x5,cat__CarName_bmw z4,cat__CarName_buick century,cat__CarName_buick century luxus (sw),cat__CarName_buick century special,cat__CarName_buick electra 225 custom,cat__CarName_buick opel isuzu deluxe,cat__CarName_buick regal sport coupe (turbo),cat__CarName_buick skyhawk,cat__CarName_buick skylark,cat__CarName_chevrolet impala,...,cat__CarName_vw dasher,cat__CarName_vw rabbit,cat__fueltype_diesel,cat__fueltype_gas,cat__aspiration_std,cat__aspiration_turbo,cat__doornumber_four,cat__doornumber_two,cat__carbody_convertible,cat__carbody_hardtop,cat__carbody_hatchback,cat__carbody_sedan,cat__carbody_wagon,cat__drivewheel_4wd,cat__drivewheel_fwd,cat__drivewheel_rwd,cat__enginelocation_front,cat__enginelocation_rear,cat__enginetype_dohc,cat__enginetype_dohcv,cat__enginetype_l,cat__enginetype_ohc,cat__enginetype_ohcf,cat__enginetype_ohcv,cat__enginetype_rotor,cat__cylindernumber_eight,cat__cylindernumber_five,cat__cylindernumber_four,cat__cylindernumber_six,cat__cylindernumber_three,cat__cylindernumber_twelve,cat__cylindernumber_two,cat__fuelsystem_1bbl,cat__fuelsystem_2bbl,cat__fuelsystem_4bbl,cat__fuelsystem_idi,cat__fuelsystem_mfi,cat__fuelsystem_mpfi,cat__fuelsystem_spdi,cat__fuelsystem_spfi
0,-1.723622,1.743470,-1.690772,-0.426521,-0.844782,-2.020417,-0.014566,0.074449,0.519071,-1.839377,-0.288349,0.174483,-0.262960,-0.646553,-0.546059,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-1.706724,1.743470,-1.690772,-0.426521,-0.844782,-2.020417,-0.014566,0.074449,0.519071,-1.839377,-0.288349,0.174483,-0.262960,-0.646553,-0.546059,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-1.689826,0.133509,-0.708596,-0.231513,-0.190566,-0.543527,0.514882,0.604046,-2.404880,0.685946,-0.288349,1.264536,-0.262960,-0.953012,-0.691627,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,-1.672928,0.938490,0.173698,0.207256,0.136542,0.235942,-0.420797,-0.431076,-0.517266,0.462183,-0.035973,-0.053668,0.787855,-0.186865,-0.109354,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-1.656029,0.938490,0.107110,0.207256,0.230001,0.235942,0.516807,0.218885,-0.517266,0.462183,-0.540725,0.275883,0.787855,-1.106241,-1.273900,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## Split train and test

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

# Model Design and Building

In [22]:
X_processed.shape

(205, 200)

## Model Architecture

```
Input Layer: Number of features = 25
Hidden Layer 1: 64 neurons (ReLU)
Hidden Layer 2: 32 neurons (ReLU)
Output Layer: 1 neuron (Price)
```

In [35]:
# Feedforward model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,505 (115.25 KB)

 Trainable params: 29,505 (115.25 KB)

 Non-trainable params: 0 (0.00 B)

## Compile the model

In [36]:
# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

## Train the model

In [37]:
# Train model
model.fit(X_train, y_train, epochs=200, batch_size=16, validation_split=0.2, verbose=0)

# Model Evaluation

In [38]:
# Test accuracy
loss = model.evaluate(X_test, y_test)
print(f'Test Loss: {loss}')

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step - loss: 9185319.0000
Test Loss: 8579033.0


In [39]:
y_pred = model.predict(X_test)

# Mean Absolute error
mae = mean_absolute_error(y_test, y_pred)

# Mean Squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean square error
rmse = root_mean_squared_error(y_test, y_pred)

# R2 score
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'R2 Score: {r2}')

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
MAE: 2005.7773673780487
MSE: 8579032.9932295
RMSE: 2928.998633190104
R2 Score: 0.8913276408062951


# Model Testing

In [40]:
sample_record = X_train.iloc[[0]]
predicted_price = model.predict(sample_record)

print(f"Sample record features:")
sample_record

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step
Sample record features:


,num__car_ID,num__symboling,num__wheelbase,num__carlength,num__carwidth,num__carheight,num__curbweight,num__enginesize,num__boreratio,num__stroke,num__compressionratio,num__horsepower,num__peakrpm,num__citympg,num__highwaympg,cat__CarName_Nissan versa,cat__CarName_alfa-romero Quadrifoglio,cat__CarName_alfa-romero giulia,cat__CarName_alfa-romero stelvio,cat__CarName_audi 100 ls,cat__CarName_audi 100ls,cat__CarName_audi 4000,cat__CarName_audi 5000,cat__CarName_audi 5000s (diesel),cat__CarName_audi fox,cat__CarName_bmw 320i,cat__CarName_bmw x1,cat__CarName_bmw x3,cat__CarName_bmw x4,cat__CarName_bmw x5,cat__CarName_bmw z4,cat__CarName_buick century,cat__CarName_buick century luxus (sw),cat__CarName_buick century special,cat__CarName_buick electra 225 custom,cat__CarName_buick opel isuzu deluxe,cat__CarName_buick regal sport coupe (turbo),cat__CarName_buick skyhawk,cat__CarName_buick skylark,cat__CarName_chevrolet impala,...,cat__CarName_vw dasher,cat__CarName_vw rabbit,cat__fueltype_diesel,cat__fueltype_gas,cat__aspiration_std,cat__aspiration_turbo,cat__doornumber_four,cat__doornumber_two,cat__carbody_convertible,cat__carbody_hardtop,cat__carbody_hatchback,cat__carbody_sedan,cat__carbody_wagon,cat__drivewheel_4wd,cat__drivewheel_fwd,cat__drivewheel_rwd,cat__enginelocation_front,cat__enginelocation_rear,cat__enginetype_dohc,cat__enginetype_dohcv,cat__enginetype_l,cat__enginetype_ohc,cat__enginetype_ohcf,cat__enginetype_ohcv,cat__enginetype_rotor,cat__cylindernumber_eight,cat__cylindernumber_five,cat__cylindernumber_four,cat__cylindernumber_six,cat__cylindernumber_three,cat__cylindernumber_twelve,cat__cylindernumber_two,cat__fuelsystem_1bbl,cat__fuelsystem_2bbl,cat__fuelsystem_4bbl,cat__fuelsystem_idi,cat__fuelsystem_mfi,cat__fuelsystem_mpfi,cat__fuelsystem_spdi,cat__fuelsystem_spfi
66,-0.608337,-0.671472,1.022697,0.07725,0.089812,0.276967,0.278074,0.170739,0.371023,1.22937,2.99254,-0.814171,-1.944265,0.88574,1.200761,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [41]:
print(f"Predicted Price for the sample record: {predicted_price[0][0]}")

Predicted Price for the sample record: 17195.091796875
